In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler

In [ ]:
def data_preprocess(rawdata: pd.DataFrame, spec_dict):
    categorical_features_values, continuous_features_values, list_drop = spec_dict.values()

    data = pd.concat([rawdata])
    data.drop(list_drop,axis=1,inplace=True, errors='ignore')
    og_samples = data[data.columns[:-1]]

    # clamping continuous values
    data_num = data.select_dtypes(include=[np.number])
    for feature in data_num.columns:
        if data_num[feature].max()>10*data_num[feature].median() and data_num[feature].max()>10 :
            data[feature] = np.where(data[feature]<data[feature].quantile(0.95), data[feature], data[feature].quantile(0.95))

    # unskew data applying log
    data_num = data.select_dtypes(include=[np.number])
    for feature in data_num.columns:
        if data_num[feature].nunique()>continuous_features_values:
            if data_num[feature].min()==0:
                data[feature] = np.log(data[feature]+1)
            else:
                data[feature] = np.log(data[feature])

    # limit non-unique value for categorical features
    data_cat = data.select_dtypes(exclude=[np.number])
    for feature in data_cat.columns:    # plot_fig(feat)
        if data_cat[feature].nunique()>categorical_features_values:
            data[feature] = np.where(data[feature].isin(data[feature].value_counts().head(categorical_features_values).index), data[feature], '-')

    # split features and labels
    samples = data[data.columns[:-1]]
    labels = data[data.columns[-1]]

    # from bool features to int
    bool_cols = samples.select_dtypes(include=bool).columns
    samples[bool_cols] = samples[bool_cols].astype(int)

    # normalize float column
    scaler = StandardScaler()
    data_num = samples.select_dtypes(include=np.number)
    samples[data_num.columns] = scaler.fit_transform(samples[data_num.columns])

    return og_samples, samples

### Data import and cleaning 

In [ ]:
dict = {
    'categorical_features_values': 6,
    'continuous_features_values': 50,
    'list_drop': [
        'id',
        'attack_cat'
    ]
}

data_tr = pd.read_csv('../datasets/UNSW_NB15_training-set.csv', delimiter=',')
rawdata, processed_data = data_preprocess(data_tr, dict)

assert  rawdata.columns.equals(processed_data.columns)

In [ ]:
for feat in rawdata.columns:
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].hist(rawdata[feat].dropna(), bins=30, edgecolor="black", alpha=0.7)
    axes[0].set_title('rawdata')
    axes[0].grid(axis="y", linestyle="--", alpha=0.7)
    axes[1].hist(processed_data[feat].dropna(), bins=30, edgecolor="black", alpha=0.7)
    axes[1].set_title('preprocessed')
    axes[1].grid(axis="y", linestyle="--", alpha=0.7)
    fig.suptitle(f'Feature: {feat}', fontsize=16)
    plt.show()
    fig.clear()

plt.clf()

In [ ]:
def print_correlation_matrix(data: pd.DataFrame):
    cols_cat = data.select_dtypes(exclude=[np.number]).columns
    if cols_cat.any():
        for col in cols_cat:
            data[col] = data[col].astype('category')
            data[col] = data[col].cat.codes
    corr = data.corr()
    
    fig, ax = plt.subplots(figsize=(12, 12))
    ax.matshow(corr, cmap='inferno')
    n = range(0, corr.columns.size)
    ax.set_xticks(n)
    ax.set_yticks(n)
    ax.set_xticklabels(corr.columns, rotation=90)
    ax.set_yticklabels(corr.columns)
    ax.xaxis.set_ticks_position('bottom')
    plt.tight_layout()
    plt.show()

    return corr

raw_corr  = print_correlation_matrix(rawdata)
proc_corr = print_correlation_matrix(processed_data)